The HRV data undergo several preprocessing steps to ensure their quality and suitability for analysis. Segments of fixed length (e.g., 1024 HRV values) are created from the HRV data, with a 50% overlap between consecutive segments to capture temporal dependencies. To generate engineered features, the open-source Python package tsfresh is used to extract relevant characteristics from the HRV data.  A feature selection process based on p-values (with a threshold of p < 0.05) is applied to retain only statistically significant features, resulting in a final set of 28 features indicative of ADHD. These selected features are then saved to a file for each patient, which will be used by the hybrid CNN-BiLSTM-Attention model.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
import os
import glob
import csv
import re
from datetime import datetime
import pandas as pd
from tsfresh.feature_extraction import extract_features, MinimalFCParameters, EfficientFCParameters
from tsfresh.feature_selection.relevance import calculate_relevance_table
from sklearn.model_selection import train_test_split
from scipy.stats import zscore


### Set global variables

In [ ]:
_DATASET_ROOT_PATH = '/kaggle/input/hyperaktiv/hyperaktiv'
_PATH_TO_GT = os.path.join(_DATASET_ROOT_PATH, "patient_info.csv")
_PATH_TO_HRV_FEATURES = os.path.join(_DATASET_ROOT_PATH, "features.csv")
_PATH_TO_HRV_RAW = '/kaggle/input/hyperaktiv/hyperaktiv/hrv_data'
_PATH_TO_HRV_FEATURES_SEGMENG = os.path.join(_DATASET_ROOT_PATH, "hrv_features_seg")


In [ ]:
def get_relevant_features(p_value_threshold):
    dataX = pd.read_csv(_PATH_TO_HRV_FEATURES, sep=";").sort_values(by="ID")
    dataY = pd.read_csv(_PATH_TO_GT, sep=";").sort_values(by="ID")

    dataX = dataX.fillna(0)

    # Remove JSON symbols from headers
    dataX = dataX.rename(columns = lambda x:re.sub('"', '', x))
    dataX = dataX.rename(columns = lambda x:re.sub(',', '', x))
    dataY = dataY.rename(columns = lambda x:re.sub('"', '', x))
    dataY = dataY.rename(columns = lambda x:re.sub(',', '', x))

    # Match X and Y data
    dataY = dataY[dataY["ID"].isin(dataX["ID"])]
    dataX = dataX[dataX["ID"].isin(dataY["ID"])]

    dataY = dataY.set_index("ID")
    dataX = dataX.set_index("ID")

    dataY = dataY["ADHD"].copy()

    # Calculate relevance using tsfresh
    p_table = calculate_relevance_table(dataX, dataY, n_jobs=7, ml_task='classification',show_warnings=True)
    p_table.head(100)
    
    # Find relevant features whose p_value<p_value_threshold
    relevant_features = p_table[p_table.p_value<p_value_threshold].feature
    return relevant_features

In [ ]:
# Test get_relevant_features(p_value_threshold)
relevant_features = get_relevant_features(0.05) #global
print("relevant features %s" %relevant_features)

In [ ]:
print(relevant_features.shape[0])

In [ ]:
# Normalize features
def normalize_features(X):
    scaler = StandardScaler()
    return scaler.fit_transform(X)

In [ ]:
# Preprocessing all patients' HRV raw data: segment, normalize, and save to .csv files

def preprocess_hrv_data(input_data_dir, output_data_dir, segment_length):

    # Loop through each raw hrv file in the directory
    for filename in os.listdir(input_data_dir):
        if filename.endswith('.csv'):

            # Preprocess HRV data for the current patient
            filepath = os.path.join(input_data_dir, filename)

            # Load the CSV file and skip the first two rows of data
            hrv_data = pd.read_csv(filepath, delimiter=';', parse_dates=['TIMESTAMP'], skiprows=[2])
           # if hrv_data['HRV'].mean() < 1000:

            hrv_data = hrv_data.iloc[2:]  # Skip the first two rows, keeping the rest
            # Replace the last value in "HRV" if it's 2000
            if hrv_data["HRV"].iloc[-1] == 2000:
                mean_hrv = hrv_data["HRV"][hrv_data["HRV"] != 2000].mean()  # Calculate mean, excluding 2000 if present
                hrv_data.loc[hrv_data.index[-1], "HRV"] = mean_hrv

            hrv_data.replace(2000, np.nan, inplace=True)  # replace 2000 with None

            hrv_data["TIMESTAMP"] = pd.to_datetime(hrv_data["TIMESTAMP"])  # Ensure "TIME" is datetime
            hrv_data.set_index("TIMESTAMP", inplace=True)  # Set "TIME" as the index

            hrv_data = hrv_data.interpolate(method='time')  # interpolate NaN values

            nan_in_hrv = hrv_data['HRV'].isnull().sum()
            print("Patient Filepath %s" % filename + "  Number of NaN values: %s" %str(nan_in_hrv))
            
            if hrv_data["HRV"].isnull().any():
                hrv_data["HRV"] = hrv_data["HRV"].fillna(hrv_data["HRV"].mean())
    
            hrv_values = hrv_data['HRV'].values
            
            features_data = []
            # Extract features for each sequence (or use the appropriate window size)
            for i in range(0, len(hrv_values) - segment_length + 1, int(segment_length/2)):  # stride - segment_length/2
                sequence = hrv_values[i:i + segment_length]
                sequence_df = pd.DataFrame(sequence, columns=["HRV"])
                sequence_df["ID"] = int(i/segment_length)
                
                feature_vector = extract_features(sequence_df, default_fc_parameters=EfficientFCParameters(), column_id="ID", column_value="HRV", n_jobs=0, show_warnings=False)
                feature_vector.columns = feature_vector.columns.str.replace('"', '', regex=False) # remove "" in column names
                feature_vector = feature_vector.loc[:, relevant_features]
                features_data.append(feature_vector)

            features_data_array = np.vstack([segment.values for segment in features_data])
            print(features_data_array.shape)
            features_data_array = normalize_features(features_data_array)

            # save segment features for each patient
            output_filepath = os.path.join(output_data_dir, "features_seg_%s" % filename)
            np.savetxt(output_filepath, features_data_array, delimiter=";")

In [ ]:
# Run preprocess_hrv_data - should only run one time for each segment length
input_data_dir = _PATH_TO_HRV_RAW
output_data_dir = _PATH_TO_HRV_FEATURES_SEGMENG
preprocess_hrv_data(input_data_dir, output_data_dir, segment_length=1024)
